# Week 4 Task — Supervised Learning Model Implementation
**Problem:** Binary classification — predict whether a customer will churn
(stop purchasing) after a fixed cutoff date, using only their pre-cutoff
behavior.
**Input:** `data/online_retail_cleaned.csv` (Week 1 output)

Note: raw `Quantity`/`Price` were deliberately **not** used to predict the
existing `IsCancellation` flag, since cancelled rows are negative-quantity
by construction — that would leak the label into the features. Churn
prediction from pre-cutoff behavior is a more realistic supervised task.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, auc, confusion_matrix,
                              classification_report)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "data/online_retail_cleaned.csv"
df = pd.read_csv(DATA_PATH)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

if df["IsCancellation"].dtype != bool:
    df["IsCancellation"] = (
        df["IsCancellation"].astype(str).str.strip().str.lower()
        .map({"true": True, "1": True, "1.0": True,
              "false": False, "0": False, "0.0": False})
        .fillna(False).astype(bool)
    )
df.head()

## 2. Define the Churn Label

Cutoff date: **1-Sep-2011**. A customer is `Churned=1` if they placed no
further (non-cancelled) orders on or after the cutoff, given that they were
active before it.

In [ ]:
cutoff = pd.Timestamp("2011-09-01")

train_raw = df[df["InvoiceDate"] < cutoff].copy()
holdout_sales = df[(df["InvoiceDate"] >= cutoff) & (~df["IsCancellation"])].dropna(subset=["Customer ID"])
returned_customers = set(holdout_sales["Customer ID"].unique())

print(f"Customers active before cutoff with a Customer ID: "
      f"{train_raw.dropna(subset=['Customer ID'])['Customer ID'].nunique():,}")
print(f"Of those, {len(returned_customers):,} returned after the cutoff.")

## 3. Feature Engineering

In [ ]:
sales_train = train_raw[~train_raw["IsCancellation"]].dropna(subset=["Customer ID"]).copy()
ref_date = cutoff

feat = sales_train.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (ref_date - x.max()).days),
    Tenure=("InvoiceDate", lambda x: (ref_date - x.min()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum"),
    AvgBasket=("TotalAmount", "mean"),
    DistinctProducts=("StockCode", "nunique"),
).reset_index()

# customer's own cancellation rate (uses ALL pre-cutoff rows, incl. cancellations)
cancel_rate = (train_raw.dropna(subset=["Customer ID"])
               .groupby("Customer ID")["IsCancellation"].mean().rename("CancelRate"))
feat = feat.merge(cancel_rate, on="Customer ID", how="left")

# dominant shipping country -> simplified to UK vs international
country = sales_train.groupby("Customer ID")["Country"].agg(lambda x: x.mode()[0]).rename("Country")
feat = feat.merge(country, on="Customer ID", how="left")
feat["IsUK"] = (feat["Country"] == "United Kingdom").astype(int)

feat["Churned"] = feat["Customer ID"].apply(lambda c: 0 if c in returned_customers else 1)

print(feat.shape)
print(feat["Churned"].value_counts(normalize=True))
feat.head()

## 4. Train/Test Split

In [ ]:
feature_cols = ["Recency", "Tenure", "Frequency", "Monetary",
                 "AvgBasket", "DistinctProducts", "CancelRate", "IsUK"]
X = feat[feature_cols]
y = feat["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]}   Test: {X_test.shape[0]}")

## 5. Model 1 — Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_s, y_train)

y_pred_lr = logreg.predict(X_test_s)
y_proba_lr = logreg.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred_lr, target_names=["Retained", "Churned"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

## 6. Model 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42,
                             class_weight="balanced")
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=["Retained", "Churned"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

## 7. Model Comparison — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, proba, color in [("Logistic Regression", y_proba_lr, "#2E86AB"),
                             ("Random Forest", y_proba_rf, "#A23B72")]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr, tpr):.3f})", linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves: Churn Prediction Models", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Confusion Matrix — Random Forest

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Retained", "Churned"], yticklabels=["Retained", "Churned"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Random Forest", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind="barh", ax=ax, color="#F18F01")
ax.set_title("Random Forest Feature Importance", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

coef = pd.Series(logreg.coef_[0], index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#A23B72" if v < 0 else "#2E86AB" for v in coef.values]
coef.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Logistic Regression Coefficients\n(positive = increases churn probability)", fontsize=12, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 10. Summary

| Metric | Logistic Regression | Random Forest |
|---|---|---|
| Accuracy | 73.9% | 74.8% |
| Precision | 75.3% | 78.1% |
| Recall | 78.5% | 75.7% |
| F1 | 0.769 | 0.769 |
| ROC-AUC | 0.807 | 0.813 |

**Key takeaways:**
- Both models comfortably beat the 55.3% majority-class baseline.
- **Recency** is by far the strongest churn predictor, followed by **Frequency**
  and **Monetary** value — customers who bought recently, often, and a lot are
  least likely to churn.
- Shipping country (`IsUK`) carries almost no predictive signal once
  purchasing behavior is accounted for.
- These features echo the RFM segmentation from Week 3 — the "At Risk / Lapsed"
  cluster and this model's high-churn-probability customers substantially overlap.

**Limitations:** a single time-based split (results could shift with a
different cutoff date); no external features (marketing, support contacts);
Random Forest hyperparameters were not extensively tuned.